In [1]:
import os
from pathlib import Path
from openai import OpenAI
import sys
import base64
from dotenv import load_dotenv
import json
import mimetypes

In [2]:
openai = OpenAI()

In [3]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY is missing from your .env file.")

In [4]:
Models = ["openai/gpt-5-nano", "openai/gpt-4.1-nano"]

In [5]:
# add the parent directory into path to import packages
sys.path.append(str(Path.cwd().parent))

from Problems.Image.image_prompts import regular_system_prompt
from Problems.Image.image_prompts import detail_system_prompt


{'Problem_1_Regular_Q': 'You are a physics expert. From the figure provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the picture.', 'Problem_1_Obvious_Q': 'You are a physics expert. From the figure provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the picture.', 'Problem_1_non_obvious_Q': 'You are a physics expert. From the figure provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the picture.', 'Problem_2_Regular_Q': 'You are a physics expert. From the figure provide a brief description and calculate the time period of oscillation for small displacements. All the necessary values required are given in the picture.', 'Problem_2_Obvious_Q': 'You are a physics expert. From the figure provide a brief description and

In [6]:
print(regular_system_prompt.keys())

dict_keys(['Problem_1_Regular_Q', 'Problem_1_Obvious_Q', 'Problem_1_non_obvious_Q', 'Problem_2_Regular_Q', 'Problem_2_Obvious_Q', 'Problem_2_non_obvious_Q', 'Problem_3_Regular_Q', 'Problem_3_Obvious_Q', 'Problem_3_non_obvious_Q'])


In [7]:
print(detail_system_prompt.keys())

dict_keys(['Problem_1_Obvious_Q', 'Problem_1_non_obvious_Q', 'Problem_2_Obvious_Q', 'Problem_2_non_obvious_Q', 'Problem_3_Obvious_Q', 'Problem_3_non_obvious_Q'])


In [8]:
# Path to image files
image_folder = Path("../Problems/Image")

# List the image filenames
file_names = sorted(file.name for file in image_folder.glob("*.png"))

print("Total number of files:::",len(file_names))
print(file_names)

Total number of files::: 9
['Problem_1_Obvious_Q.png', 'Problem_1_Regular_Q.png', 'Problem_1_non_obvious_Q.png', 'Problem_2_Obvious_Q.png', 'Problem_2_Regular_Q.png', 'Problem_2_non_obvious_Q.png', 'Problem_3_Obvious_Q.png', 'Problem_3_Regular_Q.png', 'Problem_3_non_obvious_Q.png']


In [9]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

In [ ]:
%%time
meta_data= {} ## useful for creating metadata stored as a json file. Currently an empty dictionary

for models in Models:
    for file in file_names:
        text_folder = Path("../Problems/Image")
        file_path = text_folder/ file
        file_key = file_path.stem  ## get the name of the file, for example: Problem_1_Obvious_Q

        if file_key not in regular_system_prompt:
            print(f"For {file_key} no dictionary key was found. Some error in file naming.")        
            continue

        base64_image = base64.b64encode(file_path .read_bytes()).decode("utf-8")
        mime_type = mimetypes.guess_type(str(file_path))[0]

        system_prompt = regular_system_prompt[file_key]
        response = client.responses.create(
                model=models,
                input=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "input_text",
                                "text": system_prompt,
                                },
                            {
                                "type": "input_image",
                                "image_url": f"data:{mime_type};base64,{base64_image}",
                                },
                        ],
                    }
                ],
            )

    ## Store the output in a folder
    
        model=models
        model_name=model.split("/")[-1]
        output_texts= response.output_text
        output_filename= f"{model_name}_{file_path.stem}" + "_regular_system_prompt" +".md"
    
        folder = Path.cwd().parent / "Result" / "frontier_models" / "Image"
        output_file_path = folder / output_filename
        output_file_path.write_text(output_texts, encoding="utf-8")
        
    

    ## Create metadata
    
        output_name = f"{model_name}_{file_path.stem}" + "_regular_system_prompt" +".json" ## create filename
        output_folder = Path.cwd().parent / "Result" / "frontier_models" / "Image" / "metadata"
    
    ## Get token usage (may not be provided by every API)
        
        usage = getattr(response, "usage", None)
        metadata = {
                      
            "requested_model": model,
            "returned_model": getattr(response, "model", None),
            "response_id": getattr(response, "id", None),
    
            "problem_text": file_key,
            "system_prompt": regular_system_prompt[file_key],
            "generation_settings": meta_data,
      
                    "token_usage": {
                        "input_tokens": getattr(usage, "prompt_tokens", None),
                        "output_tokens": getattr(usage, "completion_tokens", None),
                        "total_tokens": getattr(usage, "total_tokens", None),
                    },
                }
    
    ## Convert dictionary to json
    
        meta_data_file_path = output_folder / output_name
        meta_data_file_path.write_text(
            json.dumps(metadata, indent=4, ensure_ascii=False),
                     encoding="utf-8",
                                    )
        print(f"Completed::{output_name}")
    
    

CPU times: total: 844 ms
Wall time: 3min 1s


KeyboardInterrupt: 